<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Assignment_4/MSE1003_Assignment4_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Comment out the below cells when running in Colab


In [ ]:
#!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/

In [ ]:
#import os
#repo = "MSE1003H_RijaAnsari"
#print(os.listdir(repo))


In [ ]:
#%cd MSE1003H_RijaAnsari/Assignment_4

# Assignment 4: Multi-Objective Optimization and Inference


The experiment uses a Britton-Robinson buffer, so I have included the four solution volumes in the data for your reference, along with the measured pH. In practice, you would want to re-derive the HH equation to reflect the different acid properties. However for this experiment, it is fine to just use the volume ratio of all acids to the NaOH base, rather than adjusting the HH equation for all three.


In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel

from scipy.stats import norm

In [ ]:
#install botorch quietly
!pip install botorch -q

In [ ]:
# ─── Run this first to check your BoTorch version ─────────────────────────────
import botorch, gpytorch, torch
print(f"BoTorch:  {botorch.__version__}")
print(f"GPyTorch: {gpytorch.__version__}")
print(f"PyTorch:  {torch.__version__}")

In [ ]:
import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal

from gpytorch.models import ExactGP
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from gpytorch.distributions import MultitaskMultivariateNormal
from gpytorch.means import MultitaskMean, ConstantMean
from gpytorch.kernels import ScaleKernel, RBFKernel, MultitaskKernel

from botorch.models.model import Model
from botorch.posteriors.gpytorch import GPyTorchPosterior
from botorch.acquisition.objective import GenericMCObjective
from botorch.acquisition.monte_carlo import qNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf

## Surrogate Models

### for alpha

use Henderson to determine value of alpha and beta

pH = alpha + beta log (Va/Vb)

2 objective functions 
but 1 gpr predict thing
and model thing 

but 2 models in the tqdm 

Got it — this is the real active‑learning situation:
you don’t know α and β yet, and you’re sampling pH values until your surrogate model’s posterior over α and β converges.

That means you cannot train a GP directly on α and β (because they’re not observed).
Instead, you must train a GP on pH, and infer α and β as latent parameters.

This is a classic latent‑parameter inference problem, and there are two clean ways to do it in a Bayesian optimization loop.

Let me lay out the correct structure so your notebook is scientifically tight and your code is reproducible.

In [ ]:
random_seed = 1003

#read the data
data = pd.read_csv("combined_results.csv")
data

In [ ]:
#set up valid data with value of 0.00001 for vol_sodiumHydroxide for zeros
valid_data = data.copy()

#drop sodium hydroxide zeros
valid_data = valid_data[valid_data['vol_sodiumHydroxide'] > 0].reset_index(drop=True)
#valid_data.loc[0, 'vol_sodiumHydroxide'] = 0.00001
valid_data


In [ ]:
valid_data['vol_acid'] = (valid_data['vol_aceticAcid'] + 
                          valid_data['vol_phosphoricAcid'] + 
                          valid_data['vol_boricAcid'])
valid_data['vol_base']  = valid_data['vol_sodiumHydroxide'].replace(0, 1e-6)
valid_data['vol_ratio'] = valid_data['vol_acid'] / valid_data['vol_base']

valid_data

In [ ]:
# ─── Objective functions ────────────────────────────────────────────────

def objective_alpha(log10_ratio, train_y):
    """
    log10_ratio : (n,) or (n,1) tensor of log10(vol_ratio) — already log-scaled
    train_y     : (n,)   tensor of pH values
    """
    log10_ratio = log10_ratio.squeeze()
    X     = torch.stack([torch.ones_like(log10_ratio), log10_ratio], dim=1)
    theta = torch.linalg.lstsq(X, train_y.unsqueeze(1)).solution
    return theta[0].item()

def objective_beta(log10_ratio, train_y):
    log10_ratio = log10_ratio.squeeze()
    X     = torch.stack([torch.ones_like(log10_ratio), log10_ratio], dim=1)
    theta = torch.linalg.lstsq(X, train_y.unsqueeze(1)).solution
    return theta[1].item()

def get_alpha_beta_per_point(log10_ratio, train_y):
    """
    log10_ratio : (n, 1) tensor — already log10 scaled, do NOT log again
    train_y     : (n,)   tensor of pH values
    Returns     : (n, 2) tensor of (alpha, beta) LOO estimates
    """
    n = len(log10_ratio)
    results = []
    for i in range(n):
        idx   = [j for j in range(n) if j != i]
        x_sub = log10_ratio[idx]
        y_sub = train_y[idx]
        a = objective_alpha(x_sub, y_sub)
        b = objective_beta(x_sub, y_sub)
        results.append([a, b])
    return torch.tensor(results, dtype=torch.double)

In [ ]:
# ─── Rebuild training tensors cleanly ─────────────────────────────────────────
# Ensure no NaOH zero values are present
valid_data_fit = valid_data[valid_data['vol_sodiumHydroxide'] > 0].copy()
valid_data_fit['log10_vol_ratio'] = np.log10(valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide'])

print("Fitting data:")
print(valid_data_fit[['pH_measured', 'vol_acid', 'vol_sodiumHydroxide', 
                       'log10_vol_ratio', 'epit']])

train_x_joint = torch.tensor(valid_data_fit[['log10_vol_ratio']].values, dtype=torch.double)   # shape (n, 1), values should be in roughly [-1, 2]

train_y_ph = torch.tensor(valid_data_fit['pH_measured'].values, dtype=torch.double)

# LOO estimates — now correctly uses pre-computed log10 values
train_y_ab = get_alpha_beta_per_point(train_x_joint, train_y_ph)

train_y_epit = torch.tensor(valid_data_fit['epit'].values, dtype=torch.double)

train_y_joint = torch.stack([
    train_y_ab[:, 0],
    train_y_ab[:, 1],
    train_y_epit
], dim=1)   # shape (n, 3)

# Sanity check — nothing should be NaN
print(f"\ntrain_x_joint: {train_x_joint.T}")
print(f"train_y_joint:\n{train_y_joint}")
print(f"\nNaN in x: {train_x_joint.isnan().any().item()}")
print(f"NaN in y: {train_y_joint.isnan().any().item()}")

# Bounds on log10 scale — sensible headroom around observed range
log_min = float(train_x_joint.min()) - 1.0
log_max = float(train_x_joint.max()) + 1.0
bounds  = torch.tensor([[log_min], [log_max]], dtype=torch.double)
print(f"\nbounds: log10(ratio) ∈ [{log_min:.3f}, {log_max:.3f}]")
print(f"  i.e. vol_ratio  ∈ [{10**log_min:.3f}, {10**log_max:.3f}]")

In [ ]:
class JointGP(ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = MultitaskMean(ConstantMean(), num_tasks=3)

        self.covar_module = MultitaskKernel(
            ScaleKernel(RBFKernel()),
            num_tasks=3,
            rank=1
        )

    def forward(self, x):
        mean_x  = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultitaskMultivariateNormal(mean_x, covar_x)

In [ ]:
# ─── Retrain once NaN check passes ───────────────────────────────────────────
likelihood_joint = MultitaskGaussianLikelihood(num_tasks=3).double()
model_joint      = JointGP(train_x_joint, train_y_joint, likelihood_joint).double()

model_joint.train()
likelihood_joint.train()

optimizer = torch.optim.Adam(model_joint.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood_joint, model_joint)

losses = []
for i in range(300):
    optimizer.zero_grad()
    loss = -mll(model_joint(train_x_joint), train_y_joint)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (i + 1) % 50 == 0:
        print(f"Iter {i+1}/300  Loss: {loss.item():.4f}")

## Acquisition Function

In [ ]:
class JointGPBoTorchWrapper(Model):
    _num_outputs = 3

    def __init__(self, gp_model, likelihood):
        super().__init__()
        self.model      = gp_model
        self.likelihood = likelihood

    def posterior(self, X, observation_noise=True, **kwargs):
        self.model.eval()
        self.likelihood.eval()

        # ── KEY FIX: no torch.no_grad() here ──────────────────────────────
        # optimize_acqf uses L-BFGS-B which needs gradients of the acquisition
        # score w.r.t. the candidate inputs. torch.no_grad() severs the
        # computation graph and causes the RuntimeError you saw.
        # Only use no_grad() when you're doing pure inference/plotting.
        with gpytorch.settings.fast_pred_var():
            dist = (self.likelihood(self.model(X))
                    if observation_noise
                    else self.model(X))

        return GPyTorchPosterior(dist)

    def forward(self, X):
        return self.model(X)

In [ ]:
from botorch.acquisition.multi_objective.monte_carlo import (
    qNoisyExpectedHypervolumeImprovement
)
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning
)
from botorch.sampling.normal import SobolQMCNormalSampler

# ─── Reference point ──────────────────────────────────────────────────────────
# Must be worse than the worst observed value on every objective.
# EHVI measures hypervolume improvement RELATIVE to this point —
# setting it too tight (close to observed values) shrinks the
# hypervolume and makes the signal weak.
# A safe rule: observed_min - 10% of observed range per objective.

def make_ref_point(train_y, slack=0.1):
    """
    train_y : (n, 3) tensor — columns are [α, β, E_pitt]
    Returns a (3,) tensor, one ref value per objective.
    All objectives are treated as MAXIMISATION.
    """
    y_min   = train_y.min(dim=0).values
    y_range = train_y.max(dim=0).values - y_min
    return (y_min - slack * y_range).double()

ref_point = make_ref_point(train_y_joint)
print(f"Reference point — α: {ref_point[0]:.4f}, β: {ref_point[1]:.4f}, "
      f"E_pitt: {ref_point[2]:.4f}")

In [ ]:
# ─── Recompute bounds from physical acid ratio constraint ─────────────────────
# Individual acid ratio = vol_acid_each / vol_sodiumHydroxide
#                       = vol_ratio / 3   (since vol_acid_total = 3 * vol_acid_each)
#
# Constraint: 0.02 ≤ Va_each/Vb ≤ 12
# → 0.06 ≤ vol_ratio ≤ 36
# → log10(0.06) ≤ log10(vol_ratio) ≤ log10(36)

acid_ratio_min = 0.02
acid_ratio_max = 12.0

vol_ratio_min  = acid_ratio_min * 3    # 0.06
vol_ratio_max  = acid_ratio_max * 3    # 36.0

log_min = np.log10(vol_ratio_min)      # ~-1.222
log_max = np.log10(vol_ratio_max)      # ~1.556

bounds  = torch.tensor([[log_min], [log_max]], dtype=torch.double)

print(f"Individual acid ratio constraint: [{acid_ratio_min}, {acid_ratio_max}]")
print(f"Implied vol_ratio range:          [{vol_ratio_min:.3f}, {vol_ratio_max:.3f}]")
print(f"Bounds in log10 space:            [{log_min:.4f}, {log_max:.4f}]")

In [ ]:
# ─── Segment-constrained sequential qNEHVI ────────────────────────────────────
# The acquisition landscape is flat/monotone within the new physical bounds,
# causing all 4 steps to collapse to the same boundary point.
# Fix: divide [log_min, log_max] into 4 equal segments and optimise
# within each — this guarantees diverse coverage while still picking
# the best point within each region of the input space.

botorch_model = JointGPBoTorchWrapper(model_joint, likelihood_joint)

RANDOM_SEED = random_seed
candidates   = []

# Divide input range into 4 equal segments
segment_edges = np.linspace(log_min, log_max, 5)   # 5 edges → 4 segments
segments = [(segment_edges[i], segment_edges[i+1]) for i in range(4)]

print("Search segments (log10 scale → vol_ratio → Va_each/Vb):")
for i, (lo, hi) in enumerate(segments):
    print(f"  Segment {i+1}: log10 [{lo:.3f}, {hi:.3f}] "
          f"→ ratio [{10**lo:.3f}, {10**hi:.3f}] "
          f"→ Va/Vb [{10**lo/3:.3f}, {10**hi/3:.3f}]")

print()

for q_idx, (seg_lo, seg_hi) in enumerate(segments):
    torch.manual_seed(RANDOM_SEED + q_idx)

    sampler = SobolQMCNormalSampler(
        sample_shape=torch.Size([128]),
        seed=RANDOM_SEED + q_idx
    )

    # Baseline grows with previously selected points
    X_baseline = (train_x_joint if len(candidates) == 0
                  else torch.cat([
                      train_x_joint,
                      torch.tensor(candidates, dtype=torch.double).unsqueeze(1)
                  ], dim=0))

    qNEHVI = qNoisyExpectedHypervolumeImprovement(
        model          = botorch_model,
        ref_point      = ref_point,
        X_baseline     = X_baseline,
        sampler        = sampler,
        prune_baseline = False,
        cache_root     = False,
    )

    # Bounds restricted to this segment
    seg_bounds = torch.tensor([[seg_lo], [seg_hi]], dtype=torch.double)

    torch.manual_seed(RANDOM_SEED + q_idx)
    candidate, acq_val = optimize_acqf(
        acq_function = qNEHVI,
        bounds       = seg_bounds,      # ← segment-specific bounds
        q            = 1,
        num_restarts = 20,
        raw_samples  = 256,
        options      = {"batch_limit": 5, "maxiter": 200},
    )

    chosen = candidate.item()
    candidates.append(chosen)

    vr  = 10 ** chosen
    iar = vr / 3
    print(f"Segment {q_idx+1} [{10**seg_lo/3:.3f}, {10**seg_hi/3:.3f}] → "
          f"log10(r)={chosen:.4f}, vol_ratio={vr:.4f}, "
          f"Va_each/Vb={iar:.4f}, EHVI={acq_val.item():.6f}")

# ─── Convert to lab volumes ───────────────────────────────────────────────────
TOTAL_VOL = 60.0
WATER_VOL = 10.0

def log10ratio_to_volumes(log10_ratio):
    vol_ratio   = 10 ** log10_ratio
    working_vol = TOTAL_VOL - WATER_VOL
    v_base      = working_vol / (1 + vol_ratio)
    v_acid_each = (working_vol - v_base) / 3
    return {
        'log10_vol_ratio':     round(log10_ratio, 4),
        'vol_ratio':           round(vol_ratio, 4),
        'Va_each/Vb':          round(v_acid_each / max(v_base, 1e-6), 4),
        'vol_aceticAcid':      round(max(v_acid_each, 0), 2),
        'vol_phosphoricAcid':  round(max(v_acid_each, 0), 2),
        'vol_boricAcid':       round(max(v_acid_each, 0), 2),
        'vol_sodiumHydroxide': round(max(v_base, 0), 2),
        'vol_water':           round(WATER_VOL, 2),
    }

recs   = [log10ratio_to_volumes(c) for c in candidates]
rec_df = pd.DataFrame(recs)

rec_df['total_vol'] = (rec_df['vol_aceticAcid'] +
                       rec_df['vol_phosphoricAcid'] +
                       rec_df['vol_boricAcid'] +
                       rec_df['vol_sodiumHydroxide'] +
                       rec_df['vol_water'])

in_range = rec_df['Va_each/Vb'].between(acid_ratio_min, acid_ratio_max)

print("\n" + "═" * 90)
print("RECOMMENDED NEXT 4 SAMPLING POINTS")
print("═" * 90)
print(rec_df.to_string(index=False))
print(f"\nAll volumes sum to {TOTAL_VOL} mL:             {(rec_df['total_vol'] == TOTAL_VOL).all()}")
print(f"All Va_each/Vb within [{acid_ratio_min}, {acid_ratio_max}]:  {in_range.all()}")

rec_df.drop(columns='total_vol').to_csv("recommended_points_ehvi.csv", index=False)
print("Saved → recommended_points_ehvi.csv")

In [ ]:
"""# ─── Sequential qNEHVI — one point at a time ──────────────────────────────────
# Same sequential pattern as before: grow X_baseline after each selection
# so already-chosen locations are conditioned out.
#
# Note: qNEHVI has NO scalarise() function — the hypervolume IS the objective.
# The ref_point replaces the weights entirely.

botorch_model = JointGPBoTorchWrapper(model_joint, likelihood_joint)


#RANDOM_SEED = 42
RANDOM_SEED = random_seed
candidates  = []

for q_idx in range(4):
    torch.manual_seed(RANDOM_SEED + q_idx)

    sampler = SobolQMCNormalSampler(
        sample_shape=torch.Size([128]),   # 128 sufficient for 1D; increase if noisy
        seed=RANDOM_SEED + q_idx
    )

    X_baseline = (train_x_joint if len(candidates) == 0
                  else torch.cat([
                      train_x_joint,
                      torch.tensor(candidates, dtype=torch.double).unsqueeze(1)
                  ], dim=0))

    qNEHVI = qNoisyExpectedHypervolumeImprovement(
        model            = botorch_model,
        ref_point        = ref_point,      # (3,) — replaces weights
        X_baseline       = X_baseline,
        sampler          = sampler,
        prune_baseline   = False,
        # cache_root speeds up sequential calls significantly
        cache_root       = False,          # set True once everything is stable
    )

    torch.manual_seed(RANDOM_SEED + q_idx)
    candidate, acq_val = optimize_acqf(
        acq_function = qNEHVI,
        bounds       = bounds,
        q            = 1,
        num_restarts = 20,
        raw_samples  = 512,
        options      = {"batch_limit": 5, "maxiter": 200},
    )

    chosen = candidate.item()
    candidates.append(chosen)
    print(f"Step {q_idx+1}/4 — log10(ratio)={chosen:.4f}, "
          f"vol_ratio={10**chosen:.4f}, EHVI={acq_val.item():.6f}")"""

In [ ]:
"""# ─── Rebuild wrapper ──────────────────────────────────────────────────────────
botorch_model = JointGPBoTorchWrapper(model_joint, likelihood_joint)

#random_seed = 42

torch.manual_seed(random_seed)
np.random.seed(random_seed)

# ─── Rebuild objective + sampler + qNEI ──────────────────────────────────────
def scalarise(samples, X=None):
    alpha_range = (train_y_joint[:, 0].max() - train_y_joint[:, 0].min()).clamp(min=1e-6)
    beta_range  = (train_y_joint[:, 1].max() - train_y_joint[:, 1].min()).clamp(min=1e-6)
    epit_range  = (train_y_joint[:, 2].max() - train_y_joint[:, 2].min()).clamp(min=1e-6)
    w_alpha, w_beta, w_epit = 0.35, 0.35, 0.30
    return (w_alpha * samples[..., 0] / alpha_range +
            w_beta  * samples[..., 1] / beta_range  +
            w_epit  * samples[..., 2] / epit_range)

objective = GenericMCObjective(scalarise)


# ─── Sequential optimisation: pick one point at a time ───────────────────────
# Each new point is conditioned on all previously selected points,
# so the model "sees" the pending points as fantasy observations.
# This makes duplicates structurally impossible — a point already
# selected will have zero marginal value on the next iteration.

candidates = []

for q_idx in range(4):
    torch.manual_seed(random_seed + q_idx)   # different seed per step for diversity

    # Rebuild sampler and qNEI fresh each iteration
    sampler = SobolQMCNormalSampler(sample_shape=torch.Size([512]), seed=random_seed + q_idx)

    # X_baseline grows with each selected point to condition out already-chosen locations
    if len(candidates) == 0:
        X_baseline = train_x_joint
    else:
        X_baseline = torch.cat([train_x_joint,
                                 torch.tensor(candidates, dtype=torch.double).unsqueeze(1)],
                                dim=0)

    qNEI_step = qNoisyExpectedImprovement(
        model          = botorch_model,
        X_baseline     = X_baseline,   # grows each step
        sampler        = sampler,
        objective      = objective,
        prune_baseline = False,        # keep all baseline points including pending
    )

    torch.manual_seed(random_seed + q_idx)
    candidate, acq_val = optimize_acqf(
        acq_function = qNEI_step,
        bounds       = bounds,
        q            = 1,              # one point at a time
        num_restarts = 20,
        raw_samples  = 512,
        options      = {"batch_limit": 5, "maxiter": 200},
    )

    chosen = candidate.item()
    candidates.append(chosen)
    print(f"Step {q_idx+1}/4 — log10(ratio)={chosen:.4f}, "
          f"vol_ratio={10**chosen:.4f}, acq={acq_val.item():.6f}")"""


"""sampler   = SobolQMCNormalSampler(sample_shape=torch.Size([512]), seed=random_seed)

qNEI = qNoisyExpectedImprovement(
    model          = botorch_model,
    X_baseline     = train_x_joint,
    sampler        = sampler,
    objective      = objective,
    prune_baseline = True,
)

# ─── Optimise ─────────────────────────────────────────────────────────────────
candidates, acq_value = optimize_acqf(
    acq_function = qNEI,
    bounds       = bounds,
    q            = 4,
    num_restarts = 20,
    raw_samples  = 512,
    options      = {"batch_limit": 5, "maxiter": 200},
)

print(f"qNEI value: {acq_value.item():.6f}")
print(f"\nRaw candidates (log10 scale): {candidates.squeeze().tolist()}")

candidates_rounded = candidates.round(decimals=2)
unique_mask = torch.ones(len(candidates), dtype=torch.bool)
for i in range(len(candidates)):
    for j in range(i + 1, len(candidates)):
        if torch.allclose(candidates_rounded[i], candidates_rounded[j]):
            unique_mask[j] = False
            print(f"⚠ Duplicate: points {i+1} and {j+1} are identical — dropping {j+1}")

candidates_clean = candidates[unique_mask]
print(f"{len(candidates_clean)} unique candidates")"""

"""# ─── Convert to lab volumes ───────────────────────────────────────────────────
TOTAL_VOL  = 60.0
#WATER_FRAC = 0.4
WATER_VOL = 10.0

def log10ratio_to_volumes(log10_ratio):
    vol_ratio   = 10 ** log10_ratio
    #v_water     = TOTAL_VOL * WATER_FRAC
    working_vol = TOTAL_VOL - WATER_VOL
    v_base      = working_vol / (1 + vol_ratio)
    v_acid_each = (working_vol - v_base) / 3
    return {
        'log10_vol_ratio':     round(log10_ratio, 4),
        'vol_ratio':           round(vol_ratio, 4),
        'vol_aceticAcid':      round(max(v_acid_each, 0), 2),
        'vol_phosphoricAcid':  round(max(v_acid_each, 0), 2),
        'vol_boricAcid':       round(max(v_acid_each, 0), 2),
        'vol_sodiumHydroxide': round(max(v_base, 0), 2),
        'vol_water':           round(WATER_VOL, 2),
    }

recs   = [log10ratio_to_volumes(c) for c in candidates]
rec_df = pd.DataFrame(recs)

# Verify all rows sum to total volume
rec_df['total_vol'] = (rec_df['vol_aceticAcid'] +
                       rec_df['vol_phosphoricAcid'] +
                       rec_df['vol_boricAcid'] +
                       rec_df['vol_sodiumHydroxide'] +
                       rec_df['vol_water'])

print("\n" + "═" * 80)
print("RECOMMENDED NEXT 4 SAMPLING POINTS")
print("═" * 80)
print(rec_df.to_string(index=False))
print(f"\nAll volumes sum to {TOTAL_VOL} mL: {(rec_df['total_vol'] == TOTAL_VOL).all()}")

rec_df.drop(columns='total_vol').to_csv("recommended_points.csv", index=False)
print("Saved → recommended_points.csv")"""

In [ ]:
#return table of ratios of each acid to base for next 4 points to test, with columns:

def each_acid_to_base_ratio(csv_file):
    data = pd.read_csv(csv_file)
    ratios = pd.DataFrame()

    for i in range(len(data)):
        ratios.loc[i, "Va_AA/Vb"] = data.loc[i, 'vol_aceticAcid'] / data.loc[i, 'vol_sodiumHydroxide']
        ratios.loc[i, "Va_PA/Vb"] = data.loc[i, 'vol_phosphoricAcid'] / data.loc[i, 'vol_sodiumHydroxide']
        ratios.loc[i, "Va_BA/Vb"] = data.loc[i, 'vol_boricAcid'] / data.loc[i, 'vol_sodiumHydroxide']
    return ratios



In [ ]:
each_acid_to_base_ratio("recommended_points_ehvi.csv")

In [ ]:
#save the ratios to a new CSV file
ratios = each_acid_to_base_ratio("recommended_points.csv")
ratios.to_csv("acid_to_base_ratios.csv", index=False)
print("Saved → acid_to_base_ratios.csv")

# BREAK

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated

# ─── Existing data only ───────────────────────────────────────────────────────
existing_x = train_x_joint.squeeze().numpy()   # log10(ratio), shape (4,)
existing_y = train_y_joint.numpy()             # (4, 3): [α, β, E_pitt]
n          = len(existing_x)

pH_labels  = [f"pt{int(valid_data_fit['cell'].iloc[i])}\npH={valid_data_fit['pH_measured'].iloc[i]:.1f}"
              for i in range(n)]

# Pareto check
is_pareto  = is_non_dominated(train_y_joint).numpy()   # (4,) bool

face_c = ['#2196F3'] * n
edge_c = ['#2196F3'] * n

# ─── Figure ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
fig.suptitle("Existing Data — Input & Response Space with Pareto Frontier",
             fontsize=14, fontweight='bold', y=0.99)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_input = fig.add_subplot(gs[0, :])          # full-width input space
ax_ab    = fig.add_subplot(gs[1, 0])          # α vs β
ax_aep   = fig.add_subplot(gs[1, 1])          # α vs E_pitt
ax_bep   = fig.add_subplot(gs[1, 2])          # β vs E_pitt

# ─── Panel 1: Input space with GP posteriors ──────────────────────────────────
model_joint.eval()
likelihood_joint.eval()

plot_x = torch.linspace(float(bounds[0]), float(bounds[1]), 300,
                         dtype=torch.double).unsqueeze(1)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred       = likelihood_joint(model_joint(plot_x))
    plot_mu    = pred.mean.numpy()                   # (300, 3)
    lo, hi     = pred.confidence_region()
    lo, hi     = lo.numpy(), hi.numpy()

plot_x_np = plot_x.squeeze().numpy()

def norm01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

obj_labels = ['α (normalised)', 'β (normalised)', 'E_pitt (normalised)']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']
obj_ls     = ['-', '--', '-.']

for t, (lbl, col, ls) in enumerate(zip(obj_labels, obj_colors, obj_ls)):
    mu_n = norm01(plot_mu[:, t])
    lo_n = norm01(lo[:, t])
    hi_n = norm01(hi[:, t])
    ax_input.plot(plot_x_np, mu_n, color=col, lw=2, ls=ls, label=lbl, alpha=0.85)
    ax_input.fill_between(plot_x_np, lo_n, hi_n, alpha=0.12, color=col)

# Plot each existing point
for i in range(n):
    xi = existing_x[i]
    yi = norm01(plot_mu[:, 2])[np.argmin(np.abs(plot_x_np - xi))]
    ax_input.scatter(xi, yi, c=face_c[i], edgecolors=edge_c[i],
                     marker='o', s=130, linewidths=2, zorder=6)
    if is_pareto[i]:
        ax_input.scatter(xi, yi, marker='*', s=80, c='gold',
                         edgecolors='k', linewidths=0.5, zorder=7)
    offset_y = 0.13 if i % 2 == 0 else -0.17
    ax_input.annotate(pH_labels[i],
                      xy=(xi, yi),
                      xytext=(xi + 0.05, yi + offset_y),
                      fontsize=9, ha='center',
                      arrowprops=dict(arrowstyle='->', color='gray', lw=1),
                      bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

ax_input.set_xlabel("log₁₀(V_acid / V_base)", fontsize=11)
ax_input.set_ylabel("Normalised objective value", fontsize=11)
ax_input.set_title("Input Feature Space", fontsize=12, fontweight='bold')
ax_input.set_xlim([-1.75, 1.75])
ax_input.set_ylim([-0.3, 1.4])
ax_input.legend(fontsize=9, loc='upper right')
ax_input.grid(True, alpha=0.25)

# ─── Helper: response space panel ─────────────────────────────────────────────
def response_panel(ax, x_idx, y_idx, xlabel, ylabel, title):
    ax.set_title(title, fontsize=11, fontweight='bold')

    # Faded dominated, full opacity non-dominated
    for i in range(n):
        alpha = 1.0 if is_pareto[i] else 0.40
        ax.scatter(existing_y[i, x_idx], existing_y[i, y_idx],
                   c=face_c[i], edgecolors=edge_c[i],
                   marker='o', s=120, linewidths=2,
                   alpha=alpha, zorder=5)
        if is_pareto[i]:
            ax.scatter(existing_y[i, x_idx], existing_y[i, y_idx],
                       marker='*', s=70, c='gold',
                       edgecolors='k', linewidths=0.5, zorder=7)
        x_range = existing_y[:, x_idx].max() - existing_y[:, x_idx].min()
        y_range = existing_y[:, y_idx].max() - existing_y[:, y_idx].min()
        ax.annotate(pH_labels[i],
                    xy=(existing_y[i, x_idx], existing_y[i, y_idx]),
                    xytext=(existing_y[i, x_idx] + 0.03 * x_range,
                            existing_y[i, y_idx] + 0.05 * y_range),
                    fontsize=8, ha='left',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

    # Pareto staircase
    pareto_pts = existing_y[is_pareto][:, [x_idx, y_idx]]
    if len(pareto_pts) > 1:
        sorted_p = pareto_pts[np.argsort(pareto_pts[:, 0])]
        ax.step(sorted_p[:, 0], sorted_p[:, 1], where='post',
                color='gold', lw=2, ls='--', alpha=0.85,
                zorder=4, label='Pareto front')
        ax.legend(fontsize=8)

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.25)

response_panel(ax_ab,  0, 1, 'α',  'β',         'Response: α vs β')
response_panel(ax_aep, 0, 2, 'α',  'E_pitt (V)', 'Response: α vs E_pitt')
response_panel(ax_bep, 1, 2, 'β',  'E_pitt (V)', 'Response: β vs E_pitt')

#plt.savefig("pareto_existing_points.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → pareto_existing_points.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

# ─── LOO performance metrics ──────────────────────────────────────────────────
# With 4 points we simulate "active learning iterations" via LOO:
# iteration k = model trained on k points (added in order of collection)
# This shows how accuracy and uncertainty evolve as data accumulates.

n          = len(train_x_joint)
obj_names  = ['α', 'β', 'E_pitt']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']

# Storage
rmse_per_iter   = {name: [] for name in obj_names}   # (n_iters,)
std_per_iter    = {name: [] for name in obj_names}    # mean posterior std
hv_per_iter     = []                                  # hypervolume at each iter
iter_labels     = []

# Reference point for hypervolume (must be worse than all observed values)
ref_pt = make_ref_point(train_y_joint, slack=0.1)
hv_calc = Hypervolume(ref_point=ref_pt)

# Simulate iterations: train on first k points, predict on held-out remainder
for k in range(2, n + 1):   # start at 2 — need at least 2 pts for lstsq
    idx_train = list(range(k))
    idx_test  = list(range(k, n))

    x_tr = train_x_joint[idx_train]   # (k, 1)
    y_tr = train_y_joint[idx_train]   # (k, 3)
    x_te = train_x_joint[idx_test]    # (n-k, 1) — empty at last iter
    y_te = train_y_joint[idx_test]    # (n-k, 3)

    # Retrain GP on k points
    lik_k   = MultitaskGaussianLikelihood(num_tasks=3).double()
    model_k = JointGP(x_tr, y_tr, lik_k).double()
    model_k.train(); lik_k.train()
    opt_k = torch.optim.Adam(model_k.parameters(), lr=0.1)
    mll_k = gpytorch.mlls.ExactMarginalLogLikelihood(lik_k, model_k)
    for _ in range(300):
        opt_k.zero_grad()
        loss_k = -mll_k(model_k(x_tr), y_tr)
        loss_k.backward()
        opt_k.step()

    model_k.eval(); lik_k.eval()

    # ── Predictive accuracy on held-out points ────────────────────────────────
    if len(idx_test) > 0:
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_te = lik_k(model_k(x_te))
            mu_te   = pred_te.mean.numpy()     # (n-k, 3)
        y_te_np = y_te.numpy()
        for t, name in enumerate(obj_names):
            rmse = np.sqrt(np.mean((mu_te[:, t] - y_te_np[:, t])**2))
            rmse_per_iter[name].append(rmse)
    else:
        # Last iter: no held-out points — use train residuals as proxy
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_tr = lik_k(model_k(x_tr))
            mu_tr   = pred_tr.mean.numpy()
        y_tr_np = y_tr.numpy()
        for t, name in enumerate(obj_names):
            rmse = np.sqrt(np.mean((mu_tr[:, t] - y_tr_np[:, t])**2))
            rmse_per_iter[name].append(rmse)

    # ── Uncertainty: mean posterior std across dense grid ────────────────────
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_grid = lik_k(model_k(plot_x))
        std_grid  = pred_grid.variance.sqrt().numpy()   # (300, 3)
    for t, name in enumerate(obj_names):
        std_per_iter[name].append(std_grid[:, t].mean())

    # ── Hypervolume of non-dominated set so far ───────────────────────────────
    y_tr_t    = train_y_joint[idx_train]
    pareto_k  = is_non_dominated(y_tr_t)
    pareto_pts = y_tr_t[pareto_k]
    hv        = hv_calc.compute(pareto_pts)
    hv_per_iter.append(hv)

    iter_labels.append(f"k={k}")
    print(f"k={k}: RMSE α={rmse_per_iter['α'][-1]:.4f}  "
          f"β={rmse_per_iter['β'][-1]:.4f}  "
          f"E_pitt={rmse_per_iter['E_pitt'][-1]:.4f}  "
          f"HV={hv:.6f}")

iter_x = np.arange(len(iter_labels))

# ─── Figure 1: Predictive Accuracy ───────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
fig1.suptitle("Predictive Accuracy over Active-Learning Iterations\n"
              "(LOO simulation — trained on first k points, tested on remainder)",
              fontsize=13, fontweight='bold')

for t, (name, color) in enumerate(zip(obj_names, obj_colors)):
    ax = axes1[t]
    ax.plot(iter_x, rmse_per_iter[name], 'o-',
            color=color, lw=2.5, ms=9, markeredgecolor='k', markeredgewidth=0.8)
    for xi, yi in zip(iter_x, rmse_per_iter[name]):
        ax.annotate(f"{yi:.4f}", (xi, yi),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=8.5,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xticks(iter_x)
    ax.set_xticklabels(iter_labels, fontsize=10)
    ax.set_xlabel("Training set size (k)", fontsize=11)
    ax.set_ylabel("RMSE", fontsize=11)
    ax.set_title(f"Surrogate: {name}", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    # Shade last point (full data — train residual proxy)
    ax.axvspan(iter_x[-1] - 0.4, iter_x[-1] + 0.4,
               alpha=0.12, color='gray', label='Train residual\n(no held-out)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("model_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()

# ─── Figure 2: Uncertainty Evolution + Hypervolume ───────────────────────────
fig2, axes2 = plt.subplots(1, 4, figsize=(18, 4.5))
fig2.suptitle("Uncertainty Evolution & Hypervolume Improvement over Iterations",
              fontsize=13, fontweight='bold')

# Uncertainty panels (one per objective)
for t, (name, color) in enumerate(zip(obj_names, obj_colors)):
    ax = axes2[t]
    ax.plot(iter_x, std_per_iter[name], 's-',
            color=color, lw=2.5, ms=9, markeredgecolor='k', markeredgewidth=0.8)
    for xi, yi in zip(iter_x, std_per_iter[name]):
        ax.annotate(f"{yi:.4f}", (xi, yi),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=8.5,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xticks(iter_x)
    ax.set_xticklabels(iter_labels, fontsize=10)
    ax.set_xlabel("Training set size (k)", fontsize=11)
    ax.set_ylabel("Mean posterior σ", fontsize=11)
    ax.set_title(f"Uncertainty: {name}", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Target: E_pitt should flatten (low std variance across iters)
    # α, β should converge (decreasing std)
    goal = ("Goal: ↓ converge" if name in ['α', 'β']
            else "Goal: flatten (uniform σ)")
    ax.text(0.05, 0.92, goal, transform=ax.transAxes,
            fontsize=8, color=color, style='italic',
            bbox=dict(boxstyle='round', fc='white', alpha=0.6))

# Hypervolume panel (extra credit)
ax = axes2[3]
hv_arr = np.array(hv_per_iter)
hv_improvement = np.diff(hv_arr, prepend=0.0)

ax.bar(iter_x, hv_arr, color='#9C27B0', alpha=0.6,
       edgecolor='k', linewidth=0.8, label='Cumulative HV')
ax.plot(iter_x, hv_arr, 'o-', color='#4A148C',
        lw=2, ms=8, markeredgecolor='k', markeredgewidth=0.8)

ax2_twin = ax.twinx()
ax2_twin.bar(iter_x, hv_improvement, color='#FF5722', alpha=0.35,
             edgecolor='k', linewidth=0.8, label='HV improvement')
ax2_twin.set_ylabel("HV improvement (Δ)", fontsize=10, color='#FF5722')
ax2_twin.tick_params(axis='y', labelcolor='#FF5722')

for xi, yi in zip(iter_x, hv_arr):
    ax.annotate(f"{yi:.4f}", (xi, yi),
                textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xticks(iter_x)
ax.set_xticklabels(iter_labels, fontsize=10)
ax.set_xlabel("Training set size (k)", fontsize=11)
ax.set_ylabel("Hypervolume", fontsize=11)
ax.set_title("Hypervolume Improvement", fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc = 'upper left')

In [ ]:
# ─── Sample from GP posterior to get distribution of α and β predictions ──────
# Draw N samples from the trained GP at each training point and across
# a dense grid — the spread of these samples shows how uncertain the
# model is about α and β, and whether they are converging to a tight range.

N_SAMPLES  = 2000
torch.manual_seed(random_seed)

model_joint.eval()
likelihood_joint.eval()

# Dense grid across input range for sampling
sample_x = torch.linspace(float(bounds[0]), float(bounds[1]),
                           100, dtype=torch.double).unsqueeze(1)   # (100, 1)

# Draw posterior samples — shape (100, N_SAMPLES, 3)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    post        = model_joint(sample_x)
    samples     = post.sample(torch.Size([N_SAMPLES]))   # (N_SAMPLES, 100, 3)

alpha_samples = samples[:, :, 0].numpy().ravel()   # (N_SAMPLES * 100,)
beta_samples  = samples[:, :, 1].numpy().ravel()

# Global HH fit values for reference lines
alpha_global = objective_alpha(train_x_joint, train_y_ph)
beta_global  = objective_beta(train_x_joint,  train_y_ph)

# LOO estimates (the 4 training targets) for reference
alpha_loo = train_y_joint[:, 0].numpy()
beta_loo  = train_y_joint[:, 1].numpy()

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Histogram of Predicted α and β — Convergence Assessment\n"
             f"({N_SAMPLES} posterior samples × 100 input locations)",
             fontsize=13, fontweight='bold')

for ax, samples_arr, loo_vals, global_val, name, color in zip(
    axes,
    [alpha_samples, beta_samples],
    [alpha_loo,     beta_loo],
    [alpha_global,  beta_global],
    ['α (intercept)', 'β (slope)'],
    ['#2196F3',        '#4CAF50']
):
    # Main histogram
    ax.hist(samples_arr, bins=60, color=color, edgecolor='white',
            alpha=0.75, density=True, label='GP posterior samples')

    # Overlay KDE
    from scipy.stats import gaussian_kde
    kde  = gaussian_kde(samples_arr)
    xs   = np.linspace(samples_arr.min(), samples_arr.max(), 300)
    ax.plot(xs, kde(xs), color=color, lw=2.5, label='KDE')

    # Global fit reference line
    ax.axvline(global_val, color='red', lw=2.5, ls='-',
               label=f'Global fit = {global_val:.4f}')

    # LOO training point estimates
    for i, v in enumerate(loo_vals):
        ax.axvline(v, color='orange', lw=1.5, ls='--',
                   label='LOO estimates' if i == 0 else '')
        ax.text(v, ax.get_ylim()[1] * 0.02, f' pt{i+1}',
                color='darkorange', fontsize=8, rotation=90, va='bottom')

    # Convergence stats
    mu_s  = samples_arr.mean()
    std_s = samples_arr.std()
    ax.axvline(mu_s, color='black', lw=1.5, ls=':',
               label=f'Posterior mean = {mu_s:.4f}')
    ax.axvspan(mu_s - std_s, mu_s + std_s,
               alpha=0.12, color='black', label=f'±1σ = {std_s:.4f}')

    ax.set_xlabel(name, fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"Distribution of {name} predictions\n"
                 f"mean={mu_s:.4f}, std={std_s:.4f}", fontsize=11)
    ax.legend(fontsize=8.5, loc='upper right')
    ax.grid(True, alpha=0.25)

    # Convergence annotation
    cv = abs(std_s / mu_s) * 100   # coefficient of variation %
    converged = cv < 5.0
    status = f"CV = {cv:.1f}% → {'converged ✓' if converged else 'not yet converged'}"
    ax.text(0.03, 0.97, status, transform=ax.transAxes,
            fontsize=9, va='top',
            color='green' if converged else 'red',
            bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Fix y-limits after all lines drawn (axvline changes ylim dynamically)
for ax in axes:
    ax.relim()
    ax.autoscale_view()

plt.tight_layout()
plt.savefig("alpha_beta_histograms.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → alpha_beta_histograms.png")

In [ ]:
# ─── Sample α and β from GP posterior ────────────────────────────────────────
# Draw N samples of (α, β) from the joint GP posterior across the input range,
# then compute pH = α + β·log10(r) for each sample to get a distribution
# of HH curves — mean and std of this bundle is what we plot.

N_SAMPLES = 2000
torch.manual_seed(random_seed)

model_joint.eval()
likelihood_joint.eval()

# Sample at a dense grid of log10(ratio) values
sample_x = torch.linspace(float(bounds[0]), float(bounds[1]),
                           200, dtype=torch.double).unsqueeze(1)   # (200, 1)

with torch.no_grad():
    post    = model_joint(sample_x)
    samples = post.sample(torch.Size([N_SAMPLES]))   # (N_SAMPLES, 200, 3)

alpha_samp = samples[:, :, 0]   # (N_SAMPLES, 200)
beta_samp  = samples[:, :, 1]   # (N_SAMPLES, 200)

# log10(r) grid for x-axis computation
log10r_grid = sample_x.squeeze().numpy()   # (200,)
r_grid      = 10 ** log10r_grid            # actual ratio for x-axis

# HH prediction for each sample: pH = α + β·log10(r)
# log10r_grid shape (200,) broadcasts with (N_SAMPLES, 200)
pH_samples = alpha_samp.numpy() + beta_samp.numpy() * log10r_grid   # (N_SAMPLES, 200)

# Mean and std of the pH bundle
pH_mean = pH_samples.mean(axis=0)   # (200,)
pH_std  = pH_samples.std(axis=0)    # (200,)

# Global HH fit (single best-fit line over all 4 points)
alpha_global = objective_alpha(train_x_joint, train_y_ph)
beta_global  = objective_beta(train_x_joint,  train_y_ph)
pH_global    = alpha_global + beta_global * log10r_grid

# ─── Experimental data ────────────────────────────────────────────────────────
exp_r   = (valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide']).values
exp_pH  = valid_data_fit['pH_measured'].values
exp_cell = valid_data_fit['cell'].values

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

# GP mean ± std bands
ax.plot(r_grid, pH_mean, color='#2196F3', lw=2.5,
        label='GP mean prediction', zorder=4)
ax.fill_between(r_grid,
                pH_mean - 2 * pH_std,
                pH_mean + 2 * pH_std,
                alpha=0.18, color='#2196F3', label='±2σ (95% interval)')
ax.fill_between(r_grid,
                pH_mean - pH_std,
                pH_mean + pH_std,
                alpha=0.30, color='#2196F3', label='±1σ (68% interval)')

# Global least-squares HH fit
ax.plot(r_grid, pH_global, color='#FF9800', lw=2, ls='--',
        label=f'Global fit: pH = {alpha_global:.3f} + {beta_global:.3f}·log₁₀(r)',
        zorder=3)

# Experimental data points
ax.scatter(exp_r, exp_pH, c='red', s=120, zorder=6,
           edgecolors='k', linewidths=1, marker='D',
           label='Experimental data')
for i in range(len(exp_r)):
    ax.annotate(f"pt{int(exp_cell[i])}\npH={exp_pH[i]:.2f}",
                xy=(exp_r[i], exp_pH[i]),
                xytext=(exp_r[i] * 1.15, exp_pH[i] + 0.3),
                fontsize=8.5, ha='left',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

# Formatting
ax.set_xscale('log')
ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=12)
ax.set_ylabel("pH", fontsize=12)
ax.set_title("Henderson-Hasselbalch Function\nGP Prediction with Uncertainty vs Experimental Data",
             fontsize=13, fontweight='bold')
ax.set_ylim([0, 14])
ax.axhline(7, color='gray', lw=1, ls=':', alpha=0.5, label='Neutral pH = 7')
ax.legend(fontsize=9.5, loc='upper right')
ax.grid(True, which='both', alpha=0.25)

# Equation annotation
eq_text = (f"pH = α + β·log₁₀(r)\n"
           f"α = {alpha_global:.4f}  (intercept)\n"
           f"β = {beta_global:.4f}  (slope)")
ax.text(0.03, 0.97, eq_text, transform=ax.transAxes,
        fontsize=9.5, va='top', family='monospace',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

plt.tight_layout()
plt.savefig("HH_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → HH_curve.png")
print(f"Global fit: α = {alpha_global:.4f}, β = {beta_global:.4f}")

In [ ]:
# ─── GP posterior for E_pitt across input range ───────────────────────────────
N_SAMPLES = 2000
torch.manual_seed(42)

model_joint.eval()
likelihood_joint.eval()

# Dense grid over log10(ratio) range
sample_x    = torch.linspace(float(bounds[0]), float(bounds[1]),
                              200, dtype=torch.double).unsqueeze(1)  # (200, 1)
log10r_grid = sample_x.squeeze().numpy()
r_grid      = 10 ** log10r_grid

# Draw posterior samples for E_pitt (task index 2)
with torch.no_grad():
    post       = model_joint(sample_x)
    samples    = post.sample(torch.Size([N_SAMPLES]))  # (N_SAMPLES, 200, 3)

epit_samples = samples[:, :, 2].numpy()   # (N_SAMPLES, 200)
epit_mean    = epit_samples.mean(axis=0)  # (200,)
epit_std     = epit_samples.std(axis=0)   # (200,)

# ─── Experimental data ────────────────────────────────────────────────────────
exp_r    = (valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide']).values
exp_epit = valid_data_fit['epit'].values
exp_cell = valid_data_fit['cell'].values

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

# GP mean ± std bands
ax.plot(r_grid, epit_mean, color='#E53935', lw=2.5,
        label='GP mean prediction', zorder=4)
ax.fill_between(r_grid,
                epit_mean - 2 * epit_std,
                epit_mean + 2 * epit_std,
                alpha=0.18, color='#E53935', label='±2σ (95% interval)')
ax.fill_between(r_grid,
                epit_mean - epit_std,
                epit_mean + epit_std,
                alpha=0.30, color='#E53935', label='±1σ (68% interval)')

# Experimental data points
ax.scatter(exp_r, exp_epit, c='#1565C0', s=120, zorder=6,
           edgecolors='k', linewidths=1, marker='D',
           label='Experimental data')
for i in range(len(exp_r)):
    ax.annotate(f"pt{int(exp_cell[i])}\nE={exp_epit[i]:.3f}V",
                xy=(exp_r[i], exp_epit[i]),
                xytext=(exp_r[i] * 1.15, exp_epit[i] + 0.01),
                fontsize=8.5, ha='left',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

# Formatting
ax.set_xscale('log')
ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=12)
ax.set_ylabel("E_pitt  (V vs ref)", fontsize=12)
ax.set_title("Pitting Corrosion Potential E_pitt\nGP Surrogate Prediction vs Experimental Data",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9.5, loc='upper left')
ax.grid(True, which='both', alpha=0.25)

# Stats annotation
epit_obs_mean = exp_epit.mean()
epit_obs_std  = exp_epit.std()
stats_text = (f"Observed E_pitt\n"
              f"mean = {epit_obs_mean:.4f} V\n"
              f"std  = {epit_obs_std:.4f} V\n"
              f"range = [{exp_epit.min():.3f}, {exp_epit.max():.3f}] V")
ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
        fontsize=9, va='top', ha='right', family='monospace',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

# Uncertainty flatness check — E_pitt goal is uniform σ across input range
std_cv = epit_std.std() / epit_std.mean() * 100
flat_status = f"σ uniformity CV = {std_cv:.1f}%\n{'flat ✓' if std_cv < 20 else 'not yet flat — explore more'}"
ax.text(0.03, 0.03, flat_status, transform=ax.transAxes,
        fontsize=9, va='bottom',
        color='green' if std_cv < 20 else 'red',
        bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.savefig("Epitt_surrogate.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → Epitt_surrogate.png")
print(f"E_pitt uncertainty uniformity CV: {std_cv:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated

# ─── Combine existing + recommended data ──────────────────────────────────────
# Existing training points
existing_x  = train_x_joint.squeeze().numpy()          # log10(ratio)
existing_y  = train_y_joint.numpy()                    # (n, 3): [α, β, E_pitt]

# Recommended points — get GP posterior mean at each candidate
model_joint.eval()
likelihood_joint.eval()

rec_x = torch.tensor(candidates, dtype=torch.double).unsqueeze(1)  # (4, 1)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    rec_pred = likelihood_joint(model_joint(rec_x))
    rec_y    = rec_pred.mean.numpy()   # (4, 3): predicted [α, β, E_pitt]

rec_x_np = np.array(candidates)       # log10(ratio)

# All points together for Pareto analysis
all_x = np.concatenate([existing_x, rec_x_np])          # (n+4,)
all_y = np.vstack([existing_y, rec_y])                   # (n+4, 3)
n_existing = len(existing_x)
n_total    = len(all_x)

# Labels and colours — existing = filled, recommended = hollow
point_labels = ([f"E{i+1}\npH={valid_data_fit['pH_measured'].iloc[i]:.1f}"
                 for i in range(n_existing)] +
                [f"R{i+1}" for i in range(4)])
point_colors = ['#2196F3'] * n_existing + ['#FF5722'] * 4
point_markers= ['o'] * n_existing + ['D'] * 4
point_sizes  = [120] * n_existing + [100] * 4
edge_colors  = ['#2196F3'] * n_existing + ['#FF5722'] * 4
face_colors  = ['#2196F3'] * n_existing + ['none'] * 4   # hollow for recommended

# ─── Pareto non-domination check (maximise all objectives) ────────────────────
all_y_tensor   = torch.tensor(all_y, dtype=torch.double)
is_pareto_mask = is_non_dominated(all_y_tensor).numpy()   # (n+4,) bool

print("Pareto status:")
for i in range(n_total):
    label  = point_labels[i].replace('\n', ' ')
    status = 'NON-DOMINATED ★' if is_pareto_mask[i] else 'dominated'
    print(f"  Point {i+1:>2} ({label}): {status}")

In [ ]:
# ─── Figure layout ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle("Sequential Sample Selection & Pareto Frontier Evolution",
             fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

ax_input  = fig.add_subplot(gs[0, :2])   # input space — full width top left
ax_legend = fig.add_subplot(gs[0, 2])    # legend panel
ax_ab     = fig.add_subplot(gs[1, 0])    # α vs β
ax_aep    = fig.add_subplot(gs[1, 1])    # α vs E_pitt
ax_bep    = fig.add_subplot(gs[1, 2])    # β vs E_pitt

# ─── Panel 1: Input feature space (sequential order) ─────────────────────────
ax = ax_input
ax.set_title("Input Feature Space — Sequential Sample Collection",
             fontsize=12, fontweight='bold')

# GP posterior mean curve across the input range
plot_x   = torch.linspace(float(bounds[0]), float(bounds[1]), 300,
                           dtype=torch.double).unsqueeze(1)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    plot_pred    = likelihood_joint(model_joint(plot_x))
    plot_mu      = plot_pred.mean.numpy()         # (300, 3)
    plot_lo, plot_hi = plot_pred.confidence_region()

plot_x_np = plot_x.squeeze().numpy()

# Normalise each output to [0,1] for overlay on same axis
def norm01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

for t, (label, color, ls) in enumerate(zip(
        ['α (normalised)', 'β (normalised)', 'E_pitt (normalised)'],
        ['#2196F3', '#4CAF50', '#FF9800'],
        ['-', '--', '-.'])):
    mu_n = norm01(plot_mu[:, t])
    lo_n = norm01(plot_lo.numpy()[:, t])
    hi_n = norm01(plot_hi.numpy()[:, t])
    ax.plot(plot_x_np, mu_n, color=color, lw=2, ls=ls, label=label, alpha=0.8)
    ax.fill_between(plot_x_np, lo_n, hi_n, alpha=0.10, color=color)

# Plot all points in collection order with numbered arrows
for i in range(n_total):
    xi = all_x[i]
    # y position: use normalised E_pitt for placement
    yi = norm01(plot_mu[:, 2])[np.argmin(np.abs(plot_x_np - xi))]

    ax.scatter(xi, yi,
               c=face_colors[i], edgecolors=edge_colors[i],
               marker=point_markers[i], s=point_sizes[i],
               linewidths=2, zorder=6)

    # Number label with arrow
    offset = (0.05, 0.12) if i % 2 == 0 else (0.05, -0.15)
    ax.annotate(point_labels[i],
                xy=(xi, yi),
                xytext=(xi + offset[0], yi + offset[1]),
                fontsize=8, ha='center',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    # Star for Pareto points
    if is_pareto_mask[i]:
        ax.scatter(xi, yi, marker='*', s=60, c='gold',
                   edgecolors='k', linewidths=0.5, zorder=7)

ax.set_xlabel("log₁₀(V_acid / V_base)", fontsize=11)
ax.set_ylabel("Normalised objective value", fontsize=11)
ax.set_xlim([float(bounds[0]) - 0.2, float(bounds[1]) + 0.2])
ax.set_ylim([-0.25, 1.35])
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.25)

# Vertical markers for training bounds
ax.axvline(existing_x.min(), color='gray', ls=':', lw=1, alpha=0.5, label='Observed range')
ax.axvline(existing_x.max(), color='gray', ls=':', lw=1, alpha=0.5)


# ─── Legend panel ─────────────────────────────────────────────────────────────
ax = ax_legend
ax.axis('off')

legend_items = [
    (plt.scatter([], [], c='#2196F3', marker='o', s=100, label='Existing data'),
     'Existing data (E1–E{})'.format(n_existing)),
    (plt.scatter([], [], c='none', edgecolors='#FF5722', marker='D', s=100,
                 linewidths=2, label='Recommended'),
     'Recommended (R1–R4)'),
    (plt.scatter([], [], c='none', edgecolors='gray', marker='o', s=100,
                 linewidths=1, label='Dominated'),
     'Dominated point'),
    (plt.scatter([], [], marker='*', c='gold', edgecolors='k', s=150,
                 linewidths=0.5, label='Pareto'),
     'Non-dominated (Pareto ★)'),
]

for i, (_, text) in enumerate(legend_items):
    y = 0.85 - i * 0.15
    if i == 0:
        ax.scatter(0.1, y, c='#2196F3', marker='o', s=100,
                   transform=ax.transAxes, clip_on=False)
    elif i == 1:
        ax.scatter(0.1, y, c='none', edgecolors='#FF5722', marker='D', s=100,
                   linewidths=2, transform=ax.transAxes, clip_on=False)
    elif i == 2:
        ax.scatter(0.1, y, c='none', edgecolors='gray', marker='o', s=100,
                   linewidths=1, transform=ax.transAxes, clip_on=False)
    elif i == 3:
        ax.scatter(0.1, y, marker='*', c='gold', edgecolors='k', s=200,
                   linewidths=0.5, transform=ax.transAxes, clip_on=False)
    ax.text(0.22, y, text, transform=ax.transAxes,
            fontsize=10, va='center')

ax.set_title("Legend", fontsize=12, fontweight='bold')

# Summary table
summary_rows = []
for i in range(n_total):
    summary_rows.append([
        point_labels[i].replace('\n', ' '),
        f"{all_x[i]:.3f}",
        f"{all_y[i, 0]:.3f}",
        f"{all_y[i, 1]:.3f}",
        f"{all_y[i, 2]:.3f}",
        '★' if is_pareto_mask[i] else '–'
    ])

tbl = ax.table(
    cellText   = summary_rows,
    colLabels  = ['Point', 'log₁₀r', 'α', 'β', 'E_pitt', 'Pareto'],
    loc        = 'lower center',
    cellLoc    = 'center',
    bbox       = [0.0, 0.0, 1.0, 0.48]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#E3F2FD')
        cell.set_text_props(fontweight='bold')
    elif is_pareto_mask[row - 1] if row > 0 else False:
        cell.set_facecolor('#FFF9C4')

# ─── Helper: draw one response-space Pareto panel ─────────────────────────────
def plot_response_panel(ax, xi, yi, xj, yj, xlabel, ylabel,
                        title, x_idx, y_idx):
    ax.set_title(title, fontsize=11, fontweight='bold')

    for i in range(n_total):
        dominated_alpha = 1.0 if is_pareto_mask[i] else 0.45
        ax.scatter(all_y[i, x_idx], all_y[i, y_idx],
                   c=face_colors[i], edgecolors=edge_colors[i],
                   marker=point_markers[i], s=point_sizes[i],
                   linewidths=2, alpha=dominated_alpha, zorder=5)

        ax.annotate(point_labels[i],
                    xy=(all_y[i, x_idx], all_y[i, y_idx]),
                    xytext=(all_y[i, x_idx] + 0.02 * (all_y[:, x_idx].max() -
                                                        all_y[:, x_idx].min()),
                            all_y[i, y_idx] + 0.04 * (all_y[:, y_idx].max() -
                                                        all_y[:, y_idx].min())),
                    fontsize=7.5, ha='left',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

        if is_pareto_mask[i]:
            ax.scatter(all_y[i, x_idx], all_y[i, y_idx],
                       marker='*', s=60, c='gold',
                       edgecolors='k', linewidths=0.5, zorder=7)

    # Draw Pareto staircase front
    pareto_pts = all_y[is_pareto_mask][:, [x_idx, y_idx]]
    if len(pareto_pts) > 1:
        sorted_idx = np.argsort(pareto_pts[:, 0])
        px = pareto_pts[sorted_idx, 0]
        py = pareto_pts[sorted_idx, 1]
        ax.step(px, py, where='post', color='gold',
                lw=2, ls='--', alpha=0.8, label='Pareto front', zorder=4)

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

# Draw the three response panels
plot_response_panel(ax_ab,  None, None, None, None,
                    'α', 'β', 'Response Space: α vs β', 0, 1)
plot_response_panel(ax_aep, None, None, None, None,
                    'α', 'E_pitt (V)', 'Response Space: α vs E_pitt', 0, 2)
plot_response_panel(ax_bep, None, None, None, None,
                    'β', 'E_pitt (V)', 'Response Space: β vs E_pitt', 1, 2)

plt.savefig("pareto_sample_selection.png", dpi=150, bbox_inches='tight')
plt.show()
#print("Saved → pareto_sample_selection.png")